# Radiology Revenue Cycle — Synthetic Data Generation
Populates all tables in `serverless_stable_jc9zgx_catalog.radiology_revenue_cycle` with realistic synthetic data.

**Intentionally seeded DQ defects (8 traps across 6 dimensions):**
- COMPLETENESS: ~8% of paid claims missing allowed_amount
- VALIDITY: ~7% of denials use invalid CARC codes (CO-999, PR-ZZZ, OA-500, PI-300)
- CONSISTENCY: payer_name variants on claim_header vs canonical payer table
- UNIQUENESS: ~3% rebill duplicates with unreliable is_rebill flag
- TIMELINESS: ar_aging_snapshot system_a is 9 days stale
- ACCURACY (A): ~6% of claims have TC/26 component overlap with global charges (triple-count)
- ACCURACY (B): PAY-011 and PAY-012 have allowed_amount = charge_amount (clearinghouse mapping error)
- ACCURACY (C): PAY-001 (BCBS) Q1 2026 claims carry stale contracted rates in allowed_amount (12% below actual payment)

In [0]:
import random
from datetime import date, timedelta, datetime
from pyspark.sql import functions as F, Row
from pyspark.sql.types import *

CATALOG = "serverless_stable_jc9zgx_catalog"
SCHEMA = "radiology_revenue_cycle"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

random.seed(42)  # Reproducible

# Scenario date
TODAY = date(2026, 5, 15)
START_DATE = date(2025, 5, 15)  # 12 months of data

print(f"Target: {CATALOG}.{SCHEMA}")
print(f"Scenario date range: {START_DATE} to {TODAY}")

## 1. Reference / Dimension Tables

In [0]:
# 12 service locations for Pinnacle Radiology Partners
locations = [
    # 8 freestanding imaging centers
    ("LOC-001", "Pinnacle Imaging - Oak Park", "freestanding_imaging", "415 W Madison St", "Oak Park", "IL", "60302", "11", "708-555-0101", True),
    ("LOC-002", "Pinnacle Imaging - Naperville", "freestanding_imaging", "1220 E Ogden Ave", "Naperville", "IL", "60563", "11", "630-555-0102", True),
    ("LOC-003", "Pinnacle Imaging - Schaumburg", "freestanding_imaging", "890 E Higgins Rd", "Schaumburg", "IL", "60173", "11", "847-555-0103", True),
    ("LOC-004", "Pinnacle Imaging - Evanston", "freestanding_imaging", "2100 Ridge Ave", "Evanston", "IL", "60201", "11", "847-555-0104", True),
    ("LOC-005", "Pinnacle Imaging - Orland Park", "freestanding_imaging", "15300 S 94th Ave", "Orland Park", "IL", "60462", "11", "708-555-0105", True),
    ("LOC-006", "Pinnacle Imaging - Arlington Heights", "freestanding_imaging", "3040 N Arlington Heights Rd", "Arlington Heights", "IL", "60004", "11", "847-555-0106", True),
    ("LOC-007", "Pinnacle Imaging - Joliet", "freestanding_imaging", "2800 W Jefferson St", "Joliet", "IL", "60435", "11", "815-555-0107", True),
    ("LOC-008", "Pinnacle Imaging - Downers Grove", "freestanding_imaging", "1501 Ogden Ave", "Downers Grove", "IL", "60515", "11", "630-555-0108", True),
    # 3 hospital-based reading groups
    ("LOC-009", "Rush University Medical Center - Radiology", "hospital_based", "1653 W Congress Pkwy", "Chicago", "IL", "60612", "22", "312-555-0109", True),
    ("LOC-010", "Northwestern Memorial Hospital - Imaging", "hospital_based", "251 E Huron St", "Chicago", "IL", "60611", "22", "312-555-0110", True),
    ("LOC-011", "Advocate Christ Medical Center - Radiology", "hospital_based", "4440 W 95th St", "Oak Lawn", "IL", "60453", "22", "708-555-0111", True),
    # 1 mobile MRI unit
    ("LOC-012", "Pinnacle Mobile MRI Unit", "mobile", "Routes throughout Chicagoland", "Chicago", "IL", "60601", "15", "312-555-0112", True),
]

schema = StructType([
    StructField("location_id", StringType()),
    StructField("location_name", StringType()),
    StructField("location_type", StringType()),
    StructField("address_line1", StringType()),
    StructField("city", StringType()),
    StructField("state", StringType()),
    StructField("zip_code", StringType()),
    StructField("pos_code", StringType()),
    StructField("phone", StringType()),
    StructField("is_active", BooleanType()),
])

df_locations = spark.createDataFrame(locations, schema=schema)
df_locations.write.mode("overwrite").saveAsTable("service_location")
print(f"service_location: {df_locations.count()} rows")
df_locations.display()

In [0]:
# 12 contracted payers - canonical reference
payers = [
    ("PAY-001", "Blue Cross Blue Shield of Illinois", "commercial", "fee_for_service", 365, True),
    ("PAY-002", "Aetna", "commercial", "fee_for_service", 180, True),
    ("PAY-003", "UnitedHealthcare", "commercial", "fee_for_service", 365, True),
    ("PAY-004", "Medicare Part B", "medicare", "fee_for_service", 365, True),
    ("PAY-005", "Cigna", "commercial", "fee_for_service", 180, True),
    ("PAY-006", "Humana", "commercial", "capitated", 90, True),
    ("PAY-007", "Illinois Medicaid", "medicaid", "fee_for_service", 365, True),
    ("PAY-008", "TRICARE", "tricare", "fee_for_service", 365, True),
    ("PAY-009", "Workers Comp IL", "workers_comp", "fee_for_service", 120, True),
    ("PAY-010", "Self Pay", "self_pay", None, None, True),
    ("PAY-011", "Medicare Advantage - Humana", "medicare", "capitated", 365, True),
    ("PAY-012", "Molina Healthcare", "medicaid", "fee_for_service", 180, True),
]

schema = StructType([
    StructField("payer_id", StringType()),
    StructField("payer_name", StringType()),
    StructField("payer_category", StringType()),
    StructField("contract_type", StringType()),
    StructField("timely_filing_days", IntegerType()),
    StructField("is_active", BooleanType()),
])

df_payers = spark.createDataFrame(payers, schema=schema)
df_payers.write.mode("overwrite").saveAsTable("payer")
print(f"payer: {df_payers.count()} rows")
df_payers.display()

In [0]:
# 50 radiologists across specialties
first_names = ["James", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Christopher", "Daniel",
               "Matthew", "Anthony", "Mark", "Steven", "Andrew", "Paul", "Joshua", "Kenneth", "Kevin", "Brian",
               "Sarah", "Jennifer", "Lisa", "Jessica", "Emily", "Amanda", "Rachel", "Laura", "Michelle", "Stephanie",
               "Nicole", "Elizabeth", "Katherine", "Rebecca", "Maria", "Patricia", "Catherine", "Allison", "Megan", "Angela",
               "Raj", "Priya", "Wei", "Yuki", "Ahmed", "Fatima", "Carlos", "Ana", "Dmitri", "Olga"]

last_names = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez",
              "Hernandez", "Lopez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin", "Lee",
              "Patel", "Shah", "Chen", "Wang", "Kim", "Singh", "Kumar", "Ali", "Hassan", "Nguyen",
              "Thompson", "White", "Harris", "Clark", "Lewis", "Robinson", "Walker", "Young", "Allen", "King",
              "Wright", "Scott", "Torres", "Hill", "Green", "Adams", "Nelson", "Baker", "Hall", "Rivera"]

specialties = ["diagnostic"] * 20 + ["neuroradiology"] * 10 + ["interventional"] * 6 + \
              ["breast_imaging"] * 6 + ["musculoskeletal"] * 5 + ["nuclear_medicine"] * 3

location_ids = [f"LOC-{str(i).zfill(3)}" for i in range(1, 13)]

providers = []
for i in range(50):
    npi = f"1{random.randint(100000000, 999999999)}"
    name = f"{last_names[i]}, {first_names[i]}"
    credential = "MD" if random.random() < 0.9 else "DO"
    spec = specialties[i]
    loc = random.choice(location_ids)
    providers.append((npi, f"{name} {credential}", spec, credential, "Pinnacle Radiology Partners", loc, True))

schema = StructType([
    StructField("provider_npi", StringType()),
    StructField("provider_name", StringType()),
    StructField("specialty", StringType()),
    StructField("credential", StringType()),
    StructField("group_affiliation", StringType()),
    StructField("location_id", StringType()),
    StructField("is_active", BooleanType()),
])

df_providers = spark.createDataFrame(providers, schema=schema)
df_providers.write.mode("overwrite").saveAsTable("provider")
print(f"provider: {df_providers.count()} rows")

# Store NPI list for later use
provider_npis = [p[0] for p in providers]

In [0]:
# 5000 patients
patient_first_names = ["James", "Mary", "Robert", "Patricia", "John", "Jennifer", "Michael", "Linda", "David", "Barbara",
                       "William", "Elizabeth", "Richard", "Susan", "Joseph", "Jessica", "Thomas", "Sarah", "Christopher", "Karen",
                       "Charles", "Lisa", "Daniel", "Nancy", "Matthew", "Betty", "Anthony", "Margaret", "Mark", "Sandra",
                       "Donald", "Ashley", "Steven", "Dorothy", "Paul", "Kimberly", "Andrew", "Emily", "Joshua", "Donna",
                       "Kenneth", "Michelle", "Kevin", "Carol", "Brian", "Amanda", "George", "Melissa", "Timothy", "Deborah"]

patient_last_names = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez",
                      "Hernandez", "Lopez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin", "Lee",
                      "Perez", "Thompson", "White", "Harris", "Sanchez", "Clark", "Ramirez", "Lewis", "Robinson", "Walker",
                      "Young", "Allen", "King", "Wright", "Scott", "Torres", "Nguyen", "Hill", "Flores", "Green",
                      "Adams", "Nelson", "Baker", "Hall", "Rivera", "Campbell", "Mitchell", "Carter", "Roberts", "Gomez"]

il_zips = ["60601", "60602", "60603", "60604", "60605", "60606", "60607", "60608", "60609", "60610",
           "60611", "60612", "60613", "60614", "60615", "60616", "60617", "60618", "60619", "60620",
           "60302", "60304", "60201", "60202", "60515", "60516", "60563", "60564", "60173", "60174",
           "60004", "60005", "60435", "60436", "60462", "60463", "60453", "60454", "60090", "60091"]

# Payer weights: commercial 50%, medicare 25%, medicaid 15%, other 10%
payer_weights = {
    "PAY-001": 0.15, "PAY-002": 0.10, "PAY-003": 0.12, "PAY-005": 0.08, "PAY-006": 0.05,  # commercial = 50%
    "PAY-004": 0.15, "PAY-011": 0.10,  # medicare = 25%
    "PAY-007": 0.10, "PAY-012": 0.05,  # medicaid = 15%
    "PAY-008": 0.04, "PAY-009": 0.03, "PAY-010": 0.03,  # other = 10%
}
payer_ids_weighted = []
for pid, w in payer_weights.items():
    payer_ids_weighted.extend([pid] * int(w * 100))

patients = []
for i in range(1, 5001):
    patient_id = f"PAT-{str(i).zfill(5)}"
    mrn = f"{random.randint(10000000, 99999999)}"
    fname = random.choice(patient_first_names)
    lname = random.choice(patient_last_names)
    # DOB: 1940-2005
    dob_start = date(1940, 1, 1)
    dob_days = (date(2005, 12, 31) - dob_start).days
    dob = dob_start + timedelta(days=random.randint(0, dob_days))
    # Sex distribution
    sex_roll = random.random()
    sex = "M" if sex_roll < 0.48 else ("F" if sex_roll < 0.98 else "O")
    zip_code = random.choice(il_zips)
    insurance_id = f"INS-{random.randint(100000000, 999999999)}"
    primary_payer = random.choice(payer_ids_weighted)
    created_at = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 365))
    patients.append((patient_id, mrn, fname, lname, dob, sex, zip_code, insurance_id, primary_payer, created_at))

schema = StructType([
    StructField("patient_id", StringType()),
    StructField("mrn", StringType()),
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("dob", DateType()),
    StructField("sex", StringType()),
    StructField("zip_code", StringType()),
    StructField("insurance_id", StringType()),
    StructField("primary_payer_id", StringType()),
    StructField("created_at", TimestampType()),
])

df_patients = spark.createDataFrame(patients, schema=schema)
df_patients.write.mode("overwrite").saveAsTable("patient")
print(f"patient: {df_patients.count()} rows")

# Store for later
patient_ids = [p[0] for p in patients]

## 2. Encounters

In [0]:
# ~40,000 encounters across 12 months
NUM_ENCOUNTERS = 40000

# Modality distribution: CT 30%, MRI 25%, XR 20%, US 12%, MG 8%, NM 3%, IR 2%
modality_weights = ["CT"] * 30 + ["MRI"] * 25 + ["XR"] * 20 + ["US"] * 12 + ["MG"] * 8 + ["NM"] * 3 + ["IR"] * 2

# Order status: completed 92%, cancelled 4%, no_show 3%, in_progress 1%
status_weights = ["completed"] * 92 + ["cancelled"] * 4 + ["no_show"] * 3 + ["in_progress"] * 1

# Clinical indications by modality
indications = {
    "CT": ["Abdominal pain", "Chest pain rule out PE", "Headache", "Trauma evaluation", "Cancer staging", "Kidney stones"],
    "MRI": ["Low back pain", "Knee pain", "Brain lesion evaluation", "Shoulder pain", "Spine stenosis", "MS evaluation"],
    "XR": ["Chest pain", "Fracture evaluation", "Shortness of breath", "Post-op check", "Scoliosis screening"],
    "US": ["Abdominal pain", "Thyroid nodule", "Pelvic pain", "DVT evaluation", "Gallbladder evaluation"],
    "MG": ["Screening mammogram", "Diagnostic mammogram", "Palpable mass", "Follow-up prior finding"],
    "NM": ["Cardiac stress test", "Bone scan", "Thyroid uptake", "PET/CT staging"],
    "IR": ["Vascular access placement", "Biopsy", "Drain placement", "Embolization", "Angioplasty"],
}

date_range_days = (TODAY - START_DATE).days  # 365 days

encounters = []
for i in range(1, NUM_ENCOUNTERS + 1):
    enc_id = f"ENC-{str(i).zfill(6)}"
    pat_id = random.choice(patient_ids)
    prov_npi = random.choice(provider_npis)
    loc_id = random.choice(location_ids)
    study_date = START_DATE + timedelta(days=random.randint(0, date_range_days - 1))
    modality = random.choice(modality_weights)
    accession = f"ACC-{study_date.strftime('%Y%m%d')}-{str(i).zfill(5)}"
    status = random.choice(status_weights)
    referring_npi = f"1{random.randint(100000000, 999999999)}" if random.random() < 0.85 else None
    referring_name = f"Dr. {random.choice(patient_last_names)}" if referring_npi else None
    indication = random.choice(indications[modality])
    created_at = datetime.combine(study_date - timedelta(days=random.randint(0, 3)), datetime.min.time())
    
    encounters.append((enc_id, pat_id, prov_npi, loc_id, study_date, modality, accession, 
                       status, referring_npi, referring_name, indication, created_at))

schema = StructType([
    StructField("encounter_id", StringType()),
    StructField("patient_id", StringType()),
    StructField("provider_npi", StringType()),
    StructField("location_id", StringType()),
    StructField("study_date", DateType()),
    StructField("modality", StringType()),
    StructField("accession_number", StringType()),
    StructField("order_status", StringType()),
    StructField("referring_npi", StringType()),
    StructField("referring_provider_name", StringType()),
    StructField("clinical_indication", StringType()),
    StructField("created_at", TimestampType()),
])

df_encounters = spark.createDataFrame(encounters, schema=schema)
df_encounters.write.mode("overwrite").saveAsTable("encounter")
print(f"encounter: {df_encounters.count()} rows")

# Store encounter data for claims generation
enc_data = [(e[0], e[1], e[2], e[3], e[4], e[5], e[7]) for e in encounters]  # id, patient, provider, location, date, modality, status
print(f"Completed encounters (eligible for claims): {sum(1 for e in encounters if e[7] == 'completed'):,}")

## 3. Claims with Trust Traps

**Traps seeded here:**
- CONSISTENCY: `payer_name` uses variants that don't match canonical `payer.payer_name`
- COMPLETENESS: ~8% of paid claims have NULL `total_allowed_amount`
- UNIQUENESS: ~3% of pre-March-2025 claims are duplicated as rebills without proper flags

In [0]:
# ============================================================
# CLAIM HEADER GENERATION (~38,000 rows with 3 trust traps)
# ============================================================

# --- CONSISTENCY TRAP: Payer name variant mapping ---
PAYER_NAME_VARIANTS = {
    "PAY-001": ["BCBS", "Blue Cross BS", "BCBS IL", "Blue Cross Blue Shield", "BlueCross BlueShield of IL"],
    "PAY-002": ["AETNA", "Aetna Inc", "AETNA HEALTH", "CVS/Aetna"],
    "PAY-003": ["UHC", "United Healthcare", "UnitedHealth", "United HC", "UNITED HEALTHCARE"],
    "PAY-004": ["MEDICARE", "Medicare B", "CMS Medicare", "MEDICARE PART B"],
    "PAY-005": ["CIGNA", "Cigna Health", "CIGNA HEALTH"],
    "PAY-006": ["Humana", "HUMANA", "Humana Inc"],
    "PAY-007": ["IL Medicaid", "Illinois Medicaid", "ILLINOIS MEDICAID"],
    "PAY-008": ["TRICARE", "Tricare", "TRI-CARE"],
    "PAY-009": ["Workers Comp", "WC Illinois", "Workers Comp IL"],
    "PAY-010": ["Self Pay", "SELF PAY", "Self-Pay"],
    "PAY-011": ["Medicare Adv Humana", "MA - Humana", "Medicare Advantage - Humana"],
    "PAY-012": ["Molina", "Molina Healthcare", "MOLINA"],
}

# Claim status distribution: paid 72%, denied 8%, pending 12%, partial 5%, void 3%
# CHANGED: denied 12% -> 8% to lower base denial rate toward target 6-11%
claim_status_weights = ["paid"] * 72 + ["denied"] * 8 + ["pending"] * 12 + ["partial"] * 5 + ["void"] * 3

# Only completed (and some in_progress) encounters get claims
eligible_encounters = [(e[0], e[1], e[2], e[3], e[4], e[5]) for e in enc_data if e[6] in ("completed", "in_progress")]
random.shuffle(eligible_encounters)
eligible_encounters = eligible_encounters[:38000]  # Cap at ~38K

# UNIQUENESS TRAP: The is_rebill flag was only reliably populated after a system upgrade
# in October 2025. Pre-October rebills have is_rebill=false and original_claim_id=NULL.
REBILL_FLAG_CUTOFF = date(2025, 10, 1)

# COMPLETENESS TRAP: NULL allowed targets high-charge modalities
# MRI ($2800), NM ($3200), IR ($8500) get 20% NULL rate
# Other modalities get 3% NULL rate
# This creates huge denominator inflation when Report B substitutes charge_amount
HIGH_CHARGE_NULL_RATE = 0.20  # 20% of MRI/NM/IR paid claims have NULL allowed
LOW_CHARGE_NULL_RATE = 0.03   # 3% of CT/XR/US/MG paid claims have NULL allowed
HIGH_CHARGE_MODALITIES = {"MRI", "NM", "IR"}

claims = []
claim_counter = 0
rebill_originals = []  # Track claims eligible for rebill duplication

for enc_id, pat_id, prov_npi, loc_id, study_date, modality in eligible_encounters:
    claim_counter += 1
    claim_id = f"CLM-{str(claim_counter).zfill(6)}"
    
    pat_idx = int(pat_id.split("-")[1]) - 1
    payer_id = patients[pat_idx][8]  # primary_payer_id
    
    # CONSISTENCY TRAP: Use variant payer name
    payer_name_variant = random.choice(PAYER_NAME_VARIANTS[payer_id])
    
    claim_type = random.choice(["professional", "facility"]) if modality in ("IR", "NM") else "professional"
    filing_date = study_date + timedelta(days=random.randint(1, 5))
    status = random.choice(claim_status_weights)
    
    # Adjudication date for resolved claims
    if status in ("paid", "denied", "partial"):
        adjudication_date = filing_date + timedelta(days=random.randint(14, 45))
    else:
        adjudication_date = None
    
    # Generate financial amounts based on modality
    modality_charges = {"MRI": 2800, "CT": 1500, "XR": 350, "US": 600, "MG": 450, "NM": 3200, "IR": 8500}
    base_charge = modality_charges[modality]
    total_charge = round(base_charge * random.uniform(0.7, 1.3), 2)
    
    # Allowed and paid amounts
    if status in ("paid", "partial"):
        total_allowed = round(total_charge * random.uniform(0.4, 0.8), 2)
        total_paid = round(total_allowed * random.uniform(0.85, 1.0), 2)
    elif status == "denied":
        total_allowed = None
        total_paid = 0.0
    else:  # pending, void
        total_allowed = None
        total_paid = None
    
    # COMPLETENESS TRAP: NULL allowed targets high-charge modalities preferentially
    null_allowed = False
    if status == "paid":
        null_rate = HIGH_CHARGE_NULL_RATE if modality in HIGH_CHARGE_MODALITIES else LOW_CHARGE_NULL_RATE
        if random.random() < null_rate:
            total_allowed = None
            null_allowed = True
    
    is_rebill = False
    original_claim_id = None
    created_at = datetime.combine(filing_date, datetime.min.time())
    
    claims.append({
        "claim_id": claim_id,
        "encounter_id": enc_id,
        "patient_id": pat_id,
        "payer_id": payer_id,
        "payer_name": payer_name_variant,
        "rendering_npi": prov_npi,
        "billing_npi": provider_npis[0],
        "location_id": loc_id,
        "claim_type": claim_type,
        "filing_date": filing_date,
        "total_charge_amount": total_charge,
        "total_allowed_amount": total_allowed,
        "total_paid_amount": total_paid,
        "claim_status": status,
        "adjudication_date": adjudication_date,
        "is_rebill": is_rebill,
        "original_claim_id": original_claim_id,
        "created_at": created_at,
        "_modality": modality,
        "_study_date": study_date,
        "_null_allowed": null_allowed,
    })
    
    # Track pre-October 2025 paid claims for UNIQUENESS trap
    if study_date < REBILL_FLAG_CUTOFF and status == "paid":
        rebill_originals.append(claim_id)

# --- UNIQUENESS TRAP: Duplicate ~3% of pre-cutoff claims as unflagged rebills ---
num_rebills = int(len(rebill_originals) * 0.03)
rebill_candidates = random.sample(rebill_originals, min(num_rebills, len(rebill_originals)))

for orig_id in rebill_candidates:
    claim_counter += 1
    new_claim_id = f"CLM-{str(claim_counter).zfill(6)}"
    orig = next(c for c in claims if c["claim_id"] == orig_id)
    rebill_claim = orig.copy()
    rebill_claim["claim_id"] = new_claim_id
    rebill_claim["is_rebill"] = False  # NOT flagged - this is the trap
    rebill_claim["original_claim_id"] = None  # NOT linked - this is the trap
    rebill_claim["filing_date"] = orig["filing_date"] + timedelta(days=random.randint(5, 30))
    rebill_claim["created_at"] = datetime.combine(rebill_claim["filing_date"], datetime.min.time())
    claims.append(rebill_claim)

# Also add ~50 post-cutoff rebills WITH proper flags (to show the system works after upgrade)
post_cutoff_paid = [c["claim_id"] for c in claims if c["_study_date"] >= REBILL_FLAG_CUTOFF and c["claim_status"] == "paid"]
post_cutoff_rebills = random.sample(post_cutoff_paid, min(50, len(post_cutoff_paid)))

for orig_id in post_cutoff_rebills:
    claim_counter += 1
    new_claim_id = f"CLM-{str(claim_counter).zfill(6)}"
    orig = next(c for c in claims if c["claim_id"] == orig_id)
    rebill_claim = orig.copy()
    rebill_claim["claim_id"] = new_claim_id
    rebill_claim["is_rebill"] = True  # Properly flagged
    rebill_claim["original_claim_id"] = orig_id  # Properly linked
    rebill_claim["filing_date"] = orig["filing_date"] + timedelta(days=random.randint(5, 15))
    rebill_claim["created_at"] = datetime.combine(rebill_claim["filing_date"], datetime.min.time())
    claims.append(rebill_claim)

print(f"Total claims generated: {len(claims):,}")
print(f"  - Pre-Oct unflagged rebills (UNIQUENESS trap): {len(rebill_candidates)}")
print(f"  - Post-Oct flagged rebills (proper): {len(post_cutoff_rebills)}")
null_allowed_count = sum(1 for c in claims if c['_null_allowed'])
total_paid = sum(1 for c in claims if c['claim_status'] == 'paid')
print(f"  - Paid claims with NULL allowed (COMPLETENESS trap): {null_allowed_count:,} ({null_allowed_count/total_paid*100:.1f}% of paid)")
print(f"  - High-charge NULL (MRI/NM/IR): {sum(1 for c in claims if c['_null_allowed'] and c['_modality'] in HIGH_CHARGE_MODALITIES):,}")
print(f"  - Low-charge NULL (CT/XR/US/MG): {sum(1 for c in claims if c['_null_allowed'] and c['_modality'] not in HIGH_CHARGE_MODALITIES):,}")
print(f"  - Total pre-cutoff paid claims: {len(rebill_originals):,}")

# Write to table (drop internal columns)
claim_rows = [{k: v for k, v in c.items() if not k.startswith("_")} for c in claims]

schema = StructType([
    StructField("claim_id", StringType()),
    StructField("encounter_id", StringType()),
    StructField("patient_id", StringType()),
    StructField("payer_id", StringType()),
    StructField("payer_name", StringType()),
    StructField("rendering_npi", StringType()),
    StructField("billing_npi", StringType()),
    StructField("location_id", StringType()),
    StructField("claim_type", StringType()),
    StructField("filing_date", DateType()),
    StructField("total_charge_amount", DoubleType()),
    StructField("total_allowed_amount", DoubleType()),
    StructField("total_paid_amount", DoubleType()),
    StructField("claim_status", StringType()),
    StructField("adjudication_date", DateType()),
    StructField("is_rebill", BooleanType()),
    StructField("original_claim_id", StringType()),
    StructField("created_at", TimestampType()),
])

df_claims = spark.createDataFrame(claim_rows, schema=schema)
df_claims.write.mode("overwrite").saveAsTable("claim_header")
print(f"\nclaim_header written: {df_claims.count():,} rows")

In [0]:
# ============================================================
# CLAIM LINE GENERATION (~95,000 rows)
# ACCURACY TRAPS:
#   (A) TC/26 component overlap with global charges (~2% of claims) — REDUCED from 6%
#   (B) allowed_amount = charge_amount for PAY-011/PAY-012 (clearinghouse error)
#   (C) BCBS Q1 2026: allowed_amount 12% below actual payment (stale fee schedule)
# COMPLETENESS TRAP: NULL allowed on ~10% of paid claims (concentrated on high-charge)
# ============================================================

# CPT codes by modality
CPT_BY_MODALITY = {
    "MRI": ["70553", "72148", "73721", "70551", "73718", "72141"],
    "CT":  ["74177", "72131", "70553", "74178", "71260", "72192"],
    "XR":  ["71046", "73030", "72100", "73060", "70260", "72070"],
    "US":  ["76700", "76856", "93306", "76770", "76536", "76830"],
    "MG":  ["77067", "77066", "77065", "77063"],
    "NM":  ["78452", "78816", "78815", "78300"],
    "IR":  ["36247", "37228", "49083", "10005", "47000"],
}

# Charge amounts by modality (avg from design doc)
MODALITY_CHARGES = {"MRI": 2800, "CT": 1500, "XR": 350, "US": 600, "MG": 450, "NM": 3200, "IR": 8500}

# CARC codes for adjustments
CARC_CODES = ["CO-45", "CO-4", "CO-16", "CO-50", "CO-197", "PR-96", "OA-23", "PR-1", "PR-2", "CO-42"]

# Line count weights: 1=15%, 2=40%, 3=30%, 4=10%, 5=5%
line_count_weights = [1] * 15 + [2] * 40 + [3] * 30 + [4] * 10 + [5] * 5

# Build a lookup for null_allowed claims
null_allowed_claims = {c["claim_id"] for c in claims if c.get("_null_allowed", False)}

# ACCURACY (B): Payers with clearinghouse mapping error (allowed = billed)
CLEARINGHOUSE_ERROR_PAYERS = {"PAY-011", "PAY-012"}

# ACCURACY (C): BCBS stale fee schedule window
BCBS_STALE_START = date(2026, 1, 1)
BCBS_STALE_END = date(2026, 2, 28)
BCBS_RATE_INCREASE = 0.12  # 12% increase not reflected in allowed

claim_lines = []
line_counter = 0
component_overlap_claims = 0  # Counter for ACCURACY (A)

for claim in claims:
    claim_id = claim["claim_id"]
    modality = claim.get("_modality", "CT")  # fallback
    study_date = claim.get("_study_date", claim["filing_date"])
    status = claim["claim_status"]
    payer_id = claim["payer_id"]
    is_null_allowed = claim_id in null_allowed_claims
    
    num_lines = random.choice(line_count_weights)
    cpt_options = CPT_BY_MODALITY.get(modality, CPT_BY_MODALITY["CT"])
    base_charge = MODALITY_CHARGES.get(modality, 1500)
    
    # ACCURACY (A): ~2% of claims get TC/26 component overlap (REDUCED from 6%)
    # Only applies to claims with modalities that have TC/26 splits (MRI, CT, US, NM)
    add_component_overlap = False
    overlap_cpt = None
    if modality in ("MRI", "CT", "US", "NM") and random.random() < 0.02:
        add_component_overlap = True
        overlap_cpt = random.choice(cpt_options)
        component_overlap_claims += 1
    
    for line_num in range(1, num_lines + 1):
        line_counter += 1
        line_id = f"LN-{str(line_counter).zfill(7)}"
        
        cpt = random.choice(cpt_options)
        # Modifiers: professional (26), technical (TC), or none (global)
        mod_roll = random.random()
        modifier_1 = "26" if mod_roll < 0.35 else ("TC" if mod_roll < 0.65 else None)
        modifier_2 = None
        
        icd10 = random.choice(["M54.5", "R10.9", "Z12.31", "M79.3", "R06.02", "G43.909", "R07.9", "M25.561", "Z87.891", "S72.001A"])
        units = 1
        
        # Charge with variance per line (split from total)
        charge = round((base_charge / num_lines) * random.uniform(0.7, 1.3), 2)
        
        # Financial amounts depend on claim status
        if status in ("paid", "partial"):
            if is_null_allowed:
                # COMPLETENESS TRAP: Mirror header NULL allowed
                allowed = None
                paid = round(charge * random.uniform(0.3, 0.6), 2)  # Still has a payment
            elif payer_id in CLEARINGHOUSE_ERROR_PAYERS:
                # ACCURACY (B): allowed_amount = charge_amount (clearinghouse mapping error)
                allowed = charge  # This is WRONG - should be 40-60% of charge
                paid = round(charge * random.uniform(0.4, 0.6), 2)  # Actual payment is correct
            elif payer_id == "PAY-001" and BCBS_STALE_START <= study_date <= BCBS_STALE_END:
                # ACCURACY (C): BCBS Q1 2026 - allowed uses OLD rate (12% low)
                correct_allowed = round(charge * random.uniform(0.5, 0.7), 2)
                allowed = round(correct_allowed * (1 - BCBS_RATE_INCREASE), 2)  # Stale (lower) rate
                paid = round(correct_allowed * random.uniform(0.90, 1.0), 2)  # Actual payment at new rate
            else:
                allowed = round(charge * random.uniform(0.4, 0.8), 2)
                paid = round(allowed * random.uniform(0.85, 1.0), 2)
        elif status == "denied":
            allowed = None
            paid = 0.0
        else:
            allowed = None
            paid = None
        
        # Adjustment amount and CARC
        carc = random.choice(CARC_CODES) if status in ("paid", "partial", "denied") else None
        rarc = random.choice(["N362", "N522", "M15", "N479", "MA04", None]) if carc else None
        
        if allowed is not None and charge > allowed:
            adjustment = round(-(charge - allowed), 2)  # Negative (reduces revenue)
        elif status == "denied":
            adjustment = round(-charge, 2)
        else:
            adjustment = None
        
        pos_code = "11" if random.random() < 0.7 else "22"
        
        claim_lines.append((
            line_id, claim_id, line_num, cpt, modifier_1, modifier_2,
            icd10, units, charge, allowed, paid, adjustment, carc, rarc,
            pos_code, study_date
        ))
    
    # ACCURACY (A): Add overlapping component lines for ~2% of eligible claims
    # This creates a GLOBAL line (no modifier) + duplicate TC and 26 lines for the same CPT
    if add_component_overlap:
        overlap_charge = round((base_charge / num_lines) * random.uniform(0.8, 1.2), 2)
        
        # Add GLOBAL line (no modifier) - this is the "real" line
        line_counter += 1
        global_line_id = f"LN-{str(line_counter).zfill(7)}"
        global_allowed = round(overlap_charge * random.uniform(0.4, 0.8), 2) if status in ("paid", "partial") and not is_null_allowed else None
        global_paid = round(global_allowed * random.uniform(0.85, 1.0), 2) if global_allowed else (0.0 if status == "denied" else None)
        global_adj = round(-(overlap_charge - global_allowed), 2) if global_allowed and overlap_charge > global_allowed else None
        global_carc = random.choice(CARC_CODES) if status in ("paid", "partial", "denied") else None
        
        claim_lines.append((
            global_line_id, claim_id, num_lines + 1, overlap_cpt, None, None,  # No modifier = global
            "M54.5", 1, overlap_charge, global_allowed, global_paid, global_adj, global_carc, None,
            pos_code, study_date
        ))
        
        # Add DUPLICATE TC (technical) line - this is the OVERLAP
        line_counter += 1
        tc26_line_id = f"LN-{str(line_counter).zfill(7)}"
        tc_charge = round(overlap_charge * 0.6, 2)  # TC is ~60% of global
        tc_allowed = round(tc_charge * random.uniform(0.4, 0.8), 2) if status in ("paid", "partial") and not is_null_allowed else None
        tc_paid = round(tc_allowed * random.uniform(0.85, 1.0), 2) if tc_allowed else (0.0 if status == "denied" else None)
        
        claim_lines.append((
            tc26_line_id, claim_id, num_lines + 2, overlap_cpt, "TC", None,
            "M54.5", 1, tc_charge, tc_allowed, tc_paid, None, global_carc, None,
            pos_code, study_date
        ))
        
        # Add DUPLICATE 26 (professional) line - this is also OVERLAP
        line_counter += 1
        prof_line_id = f"LN-{str(line_counter).zfill(7)}"
        prof_charge = round(overlap_charge * 0.4, 2)  # 26 is ~40% of global
        prof_allowed = round(prof_charge * random.uniform(0.4, 0.8), 2) if status in ("paid", "partial") and not is_null_allowed else None
        prof_paid = round(prof_allowed * random.uniform(0.85, 1.0), 2) if prof_allowed else (0.0 if status == "denied" else None)
        
        claim_lines.append((
            prof_line_id, claim_id, num_lines + 3, overlap_cpt, "26", None,
            "M54.5", 1, prof_charge, prof_allowed, prof_paid, None, global_carc, None,
            pos_code, study_date
        ))

# --- Summary ---
print(f"claim_line rows generated: {len(claim_lines):,}")
print(f"\nACCURACY TRAP (A) - TC/26 Component Overlap:")
print(f"  Claims with overlap: {component_overlap_claims:,} ({component_overlap_claims/len(claims)*100:.1f}%)")

print(f"\nACCURACY TRAP (B) - Clearinghouse Mapping (allowed=billed):")
b_count = sum(1 for l in claim_lines if l[9] is not None and l[9] == l[8])  # allowed == charge
print(f"  Lines with allowed=charge: {b_count:,}")

print(f"\nACCURACY TRAP (C) - BCBS Stale Fee Schedule:")
c_count = sum(1 for i, c in enumerate(claims) if c['payer_id'] == 'PAY-001' and c.get('_study_date') and BCBS_STALE_START <= c['_study_date'] <= BCBS_STALE_END and c['claim_status'] in ('paid','partial'))
print(f"  BCBS claims in stale window (Jan-Feb 2026): {c_count:,}")

print(f"\nCOMPLETENESS TRAP - NULL allowed on paid:")
null_count = sum(1 for l in claim_lines if l[9] is None and l[10] is not None and l[10] > 0)
print(f"  Lines with NULL allowed but paid > 0: {null_count:,}")

# Write to table
schema = StructType([
    StructField("claim_line_id", StringType()),
    StructField("claim_id", StringType()),
    StructField("line_number", IntegerType()),
    StructField("cpt_code", StringType()),
    StructField("modifier_1", StringType()),
    StructField("modifier_2", StringType()),
    StructField("icd10_code", StringType()),
    StructField("units", IntegerType()),
    StructField("charge_amount", DoubleType()),
    StructField("allowed_amount", DoubleType()),
    StructField("paid_amount", DoubleType()),
    StructField("adjustment_amount", DoubleType()),
    StructField("adjustment_reason_code", StringType()),
    StructField("remark_code", StringType()),
    StructField("place_of_service", StringType()),
    StructField("service_date", DateType()),
])

df_lines = spark.createDataFrame(claim_lines, schema=schema)
df_lines.write.mode("overwrite").saveAsTable("claim_line")
print(f"\nclaim_line written: {df_lines.count():,} rows")

## 4. Denials with VALIDITY Trap

Invalid CARC codes planted: CO-999, PR-ZZZ, OA-500, PI-300 — these don't exist in the X12 835 standard.

In [0]:
# ============================================================
# DENIAL GENERATION with VALIDITY trap
# CHANGED: Base denied rate reduced from 12% to 8% (in claim_header)
# CHANGED: Invalid CARC rate increased to 30% AND those claims ONLY get invalid denials
#          This ensures excluding invalid CARCs removes those claims entirely from denied count
# ============================================================

# Valid CARC codes with realistic radiology distribution
VALID_CARC_WEIGHTED = (
    ["CO-197"] * 22 +  # Precert/auth not obtained (top radiology denial)
    ["CO-4"] * 18 +    # Procedure inconsistent with modifier
    ["CO-16"] * 15 +   # Missing/incomplete information
    ["CO-45"] * 12 +   # Charges exceed fee schedule
    ["CO-50"] * 10 +   # Non-covered service
    ["PR-96"] * 8 +    # Non-covered charge
    ["OA-23"] * 8      # Impact of prior payer
)

# VALIDITY TRAP: Invalid CARC codes (~30% of denied claims)
# These are fake codes that don't exist in the X12 835 standard
INVALID_CARC_CODES = ["CO-999"] * 5 + ["PI-300"] * 3 + ["OA-500"] * 2  # weighted distribution

# Denial categories
CARC_TO_CATEGORY = {
    "CO-197": "auth",
    "CO-4": "coding",
    "CO-16": "coding",
    "CO-45": "coding",
    "CO-50": "medical_necessity",
    "PR-96": "eligibility",
    "OA-23": "eligibility",
    "CO-999": "unclassified",  # Invalid code
    "PI-300": "unclassified",  # Invalid code
    "OA-500": "unclassified",  # Invalid code
}

# Appeal status: none 50%, submitted 25%, in_review 10%, overturned 10%, upheld 5%
appeal_weights = ["none"] * 50 + ["submitted"] * 25 + ["in_review"] * 10 + ["overturned"] * 10 + ["upheld"] * 5

# Get denied/partial claims
denied_claims = [c for c in claims if c["claim_status"] in ("denied", "partial")]
print(f"Claims eligible for denials: {len(denied_claims):,}")

# Split into two groups:
# 30% will ONLY get invalid CARC codes (single denial per claim)
# 70% will get valid CARC codes (1-2 denials per claim)
random.shuffle(denied_claims)
invalid_split = int(len(denied_claims) * 0.30)
invalid_carc_claims = denied_claims[:invalid_split]
valid_carc_claims = denied_claims[invalid_split:]

print(f"  - Claims assigned INVALID CARCs only: {len(invalid_carc_claims):,} ({invalid_split/len(denied_claims)*100:.0f}%)")
print(f"  - Claims assigned valid CARCs: {len(valid_carc_claims):,}")

denials = []
denial_counter = 0

# --- INVALID CARC claims: exactly 1 denial per claim with an invalid code ---
for claim in invalid_carc_claims:
    denial_counter += 1
    denial_id = f"DEN-{str(denial_counter).zfill(5)}"
    
    carc = random.choice(INVALID_CARC_CODES)
    category = CARC_TO_CATEGORY.get(carc, "unclassified")
    rarc = random.choice(["N362", "N522", "M15", "N479", "MA04", "N657"])
    
    claim_line_id = None
    if random.random() < 0.6:
        matching_lines = [l[0] for l in claim_lines if l[1] == claim["claim_id"]]
        if matching_lines:
            claim_line_id = random.choice(matching_lines)
    
    denial_date = claim["adjudication_date"] if claim["adjudication_date"] else claim["filing_date"] + timedelta(days=random.randint(14, 45))
    denied_amount = round(claim["total_charge_amount"] * random.uniform(0.3, 1.0), 2)
    
    appeal_status = random.choice(appeal_weights)
    appeal_date = None
    overturn_date = None
    overturn_amount = None
    
    if appeal_status != "none":
        appeal_date = denial_date + timedelta(days=random.randint(5, 30))
    if appeal_status == "overturned":
        overturn_date = appeal_date + timedelta(days=random.randint(30, 90))
        overturn_amount = round(denied_amount * random.uniform(0.5, 1.0), 2)
    
    denials.append((
        denial_id, claim["claim_id"], claim_line_id, denial_date,
        carc, rarc, category, denied_amount,
        appeal_status, appeal_date, overturn_date, overturn_amount
    ))

# --- VALID CARC claims: 1-2 denials per claim ---
for claim in valid_carc_claims:
    num_denials = 1 if random.random() < 0.75 else 2
    
    for _ in range(num_denials):
        denial_counter += 1
        denial_id = f"DEN-{str(denial_counter).zfill(5)}"
        
        carc = random.choice(VALID_CARC_WEIGHTED)
        category = CARC_TO_CATEGORY.get(carc, "unclassified")
        rarc = random.choice(["N362", "N522", "M15", "N479", "MA04", "N657"])
        
        claim_line_id = None
        if random.random() < 0.6:
            matching_lines = [l[0] for l in claim_lines if l[1] == claim["claim_id"]]
            if matching_lines:
                claim_line_id = random.choice(matching_lines)
        
        denial_date = claim["adjudication_date"] if claim["adjudication_date"] else claim["filing_date"] + timedelta(days=random.randint(14, 45))
        denied_amount = round(claim["total_charge_amount"] * random.uniform(0.3, 1.0) / num_denials, 2)
        
        appeal_status = random.choice(appeal_weights)
        appeal_date = None
        overturn_date = None
        overturn_amount = None
        
        if appeal_status != "none":
            appeal_date = denial_date + timedelta(days=random.randint(5, 30))
        if appeal_status == "overturned":
            overturn_date = appeal_date + timedelta(days=random.randint(30, 90))
            overturn_amount = round(denied_amount * random.uniform(0.5, 1.0), 2)
        
        denials.append((
            denial_id, claim["claim_id"], claim_line_id, denial_date,
            carc, rarc, category, denied_amount,
            appeal_status, appeal_date, overturn_date, overturn_amount
        ))

print(f"\nDenials generated: {len(denials):,}")
invalid_count = sum(1 for d in denials if d[4] in ("CO-999", "PI-300", "OA-500", "PR-ZZZ"))
valid_count = len(denials) - invalid_count
print(f"  - Invalid CARC codes (VALIDITY trap): {invalid_count} ({invalid_count/len(denials)*100:.1f}%)")
print(f"  - Valid CARC codes: {valid_count}")
print(f"  - Distinct claims with ANY denial: {len(set(d[1] for d in denials)):,}")
print(f"  - Distinct claims with ONLY invalid CARCs: {len(invalid_carc_claims):,}")
print(f"  - Report A denial rate ~ {len(valid_carc_claims)/len(claims)*100:.1f}% vs Report B ~ {len(denied_claims)/len(claims)*100:.1f}%")

schema = StructType([
    StructField("denial_id", StringType()),
    StructField("claim_id", StringType()),
    StructField("claim_line_id", StringType()),
    StructField("denial_date", DateType()),
    StructField("carc_code", StringType()),
    StructField("rarc_code", StringType()),
    StructField("denial_category", StringType()),
    StructField("denied_amount", DoubleType()),
    StructField("appeal_status", StringType()),
    StructField("appeal_date", DateType()),
    StructField("overturn_date", DateType()),
    StructField("overturn_amount", DoubleType()),
])

df_denials = spark.createDataFrame(denials, schema=schema)
df_denials.write.mode("overwrite").saveAsTable("denial")
print(f"denial written: {df_denials.count():,} rows")

## 5. Payments

In [0]:
# ============================================================
# PAYMENT GENERATION (~85,000 rows)
# Payments prove claims were paid even when allowed_amount is NULL
# ============================================================

# Get paid/partial claims with their adjudication dates
paid_claims_lookup = {c["claim_id"]: c for c in claims if c["claim_status"] in ("paid", "partial")}

# Get claim lines for paid claims
paid_lines = [(l[0], l[1], l[10], l[8]) for l in claim_lines 
              if l[1] in paid_claims_lookup and l[10] is not None and l[10] > 0]  # line_id, claim_id, paid_amount, charge

print(f"Paid claim lines eligible for payments: {len(paid_lines):,}")

payments = []
pmt_counter = 0

for line_id, claim_id, paid_amount, charge_amount in paid_lines:
    claim = paid_claims_lookup[claim_id]
    adj_date = claim["adjudication_date"]
    if adj_date is None:
        adj_date = claim["filing_date"] + timedelta(days=30)
    
    pmt_counter += 1
    payment_id = f"PMT-{str(pmt_counter).zfill(6)}"
    
    payment_date = adj_date + timedelta(days=random.randint(3, 15))
    payer_id = claim["payer_id"]
    
    # Primary payment
    pmt_amount = paid_amount
    adj_amount = round(charge_amount - paid_amount, 2) if charge_amount else None
    adj_code = random.choice(["CO-45", "CO-42", "PR-1", "PR-2", None])
    check_num = f"CHK-{random.randint(1000000, 9999999)}"
    era_id = f"ERA-{payment_date.strftime('%Y%m%d')}-{random.randint(1000, 9999)}"
    
    payments.append((
        payment_id, claim_id, line_id, payer_id,
        payment_date, pmt_amount, adj_amount, adj_code,
        check_num, era_id
    ))
    
    # ~10% get a secondary payment (crossover, patient responsibility)
    if random.random() < 0.10:
        pmt_counter += 1
        sec_pmt_id = f"PMT-{str(pmt_counter).zfill(6)}"
        sec_date = payment_date + timedelta(days=random.randint(10, 30))
        sec_amount = round(paid_amount * random.uniform(0.05, 0.15), 2)
        sec_payer = random.choice(["PAY-010", claim["payer_id"]])  # self-pay or same payer
        
        payments.append((
            sec_pmt_id, claim_id, line_id, sec_payer,
            sec_date, sec_amount, None, None,
            f"CHK-{random.randint(1000000, 9999999)}",
            f"ERA-{sec_date.strftime('%Y%m%d')}-{random.randint(1000, 9999)}"
        ))

print(f"Payments generated: {len(payments):,}")

schema = StructType([
    StructField("payment_id", StringType()),
    StructField("claim_id", StringType()),
    StructField("claim_line_id", StringType()),
    StructField("payer_id", StringType()),
    StructField("payment_date", DateType()),
    StructField("payment_amount", DoubleType()),
    StructField("adjustment_amount", DoubleType()),
    StructField("adjustment_code", StringType()),
    StructField("check_number", StringType()),
    StructField("era_id", StringType()),
])

df_payments = spark.createDataFrame(payments, schema=schema)
df_payments.write.mode("overwrite").saveAsTable("payment")
print(f"payment written: {df_payments.count():,} rows")

## 6. AR Aging Snapshot with TIMELINESS Trap

- **system_b** (freestanding + mobile): snapshots up to 2026-05-15 (today)
- **system_a** (hospital-based): snapshots STOP at 2026-05-06 (9 days stale)

In [0]:
# ============================================================
# AR AGING SNAPSHOT with TIMELINESS trap
# CHANGED: Hospital (system_a) skews heavily toward 'current' bucket (low days)
#          Freestanding (system_b) skews toward older buckets (high days)
#          When Report B excludes stale hospital data, weighted avg days jump
# ============================================================

# Location classification
SYSTEM_B_LOCATIONS = [f"LOC-{str(i).zfill(3)}" for i in range(1, 9)] + ["LOC-012"]  # Freestanding + mobile
SYSTEM_A_LOCATIONS = ["LOC-009", "LOC-010", "LOC-011"]  # Hospital-based (STALE)

# TIMELINESS TRAP:
# system_b: snapshots through TODAY (2026-05-15)
# system_a: snapshots STOP at 2026-05-06 (9 days stale)
SYSTEM_B_END = TODAY  # 2026-05-15
SYSTEM_A_END = date(2026, 5, 6)  # 9 days behind
SNAPSHOT_START = TODAY - timedelta(days=30)  # 30 days of history

# Aging buckets
BUCKETS = ["current", "31_60", "61_90", "91_120", "121_plus"]

# DIFFERENTIATED AR PROFILES:
# Hospital-based (system_a): heavily weighted toward 'current' bucket
# This means they pull the overall weighted avg DOWN (lower days)
# When excluded, the remaining data has higher avg days
HOSPITAL_BUCKET_BASE = {
    "current": 120000,    # HUGE current bucket (hospitals have fast payment)
    "31_60": 15000,
    "61_90": 5000,
    "91_120": 2000,
    "121_plus": 1000,
}  # Weighted avg days: ~22 days

# Freestanding (system_b): more evenly distributed, skews older
# Without hospital data diluting the average, these show higher days
FREESTANDING_BUCKET_BASE = {
    "current": 25000,
    "31_60": 30000,       # Peak in 31-60
    "61_90": 20000,
    "91_120": 12000,
    "121_plus": 8000,
}  # Weighted avg days: ~52 days

# Select top payers for AR (not all payers have AR at every location)
ar_payer_ids = ["PAY-001", "PAY-002", "PAY-003", "PAY-004", "PAY-007", "PAY-011"]

ar_snapshots = []

for loc_id in SYSTEM_B_LOCATIONS + SYSTEM_A_LOCATIONS:
    is_hospital = loc_id in SYSTEM_A_LOCATIONS
    source = "system_a" if is_hospital else "system_b"
    end_date = SYSTEM_A_END if is_hospital else SYSTEM_B_END
    bucket_base = HOSPITAL_BUCKET_BASE if is_hospital else FREESTANDING_BUCKET_BASE
    
    # Generate daily snapshots from start to end
    current_date = SNAPSHOT_START
    while current_date <= end_date:
        # Each location has 2-4 payers with outstanding AR on any given day
        active_payers = random.sample(ar_payer_ids, random.randint(2, 4))
        
        for payer_id in active_payers:
            for bucket in BUCKETS:
                base = bucket_base[bucket]
                # Add daily variance
                outstanding = round(base * random.uniform(0.7, 1.3), 2)
                claim_count = max(1, int(outstanding / random.uniform(800, 2000)))
                
                ar_snapshots.append((
                    current_date, payer_id, loc_id, bucket,
                    claim_count, outstanding, source
                ))
        
        current_date += timedelta(days=1)

print(f"AR aging snapshots generated: {len(ar_snapshots):,}")
print(f"  - system_a (hospital, stale): snapshots through {SYSTEM_A_END}")
print(f"  - system_b (freestanding): snapshots through {SYSTEM_B_END}")
print(f"  - Gap: {(SYSTEM_B_END - SYSTEM_A_END).days} days (TIMELINESS trap)")

# Show expected AR days by source
hosp_total_weighted = sum(r[5] * {"current":15,"31_60":45,"61_90":75,"91_120":105,"121_plus":150}[r[3]] for r in ar_snapshots if r[6]=="system_a")
hosp_total_amt = sum(r[5] for r in ar_snapshots if r[6]=="system_a")
free_total_weighted = sum(r[5] * {"current":15,"31_60":45,"61_90":75,"91_120":105,"121_plus":150}[r[3]] for r in ar_snapshots if r[6]=="system_b")
free_total_amt = sum(r[5] for r in ar_snapshots if r[6]=="system_b")
print(f"\n  Expected weighted days in AR:")
print(f"    Hospital only (system_a): {hosp_total_weighted/hosp_total_amt:.0f} days")
print(f"    Freestanding only (system_b): {free_total_weighted/free_total_amt:.0f} days")
print(f"    Combined (all): {(hosp_total_weighted+free_total_weighted)/(hosp_total_amt+free_total_amt):.0f} days")

schema = StructType([
    StructField("snapshot_date", DateType()),
    StructField("payer_id", StringType()),
    StructField("location_id", StringType()),
    StructField("aging_bucket", StringType()),
    StructField("claim_count", IntegerType()),
    StructField("outstanding_amount", DoubleType()),
    StructField("snapshot_source", StringType()),
])

df_ar = spark.createDataFrame(ar_snapshots, schema=schema)
df_ar.write.mode("overwrite").saveAsTable("ar_aging_snapshot")
print(f"\nar_aging_snapshot written: {df_ar.count():,} rows")

## 7. Authorizations

Prior auths for advanced imaging. No hard FK to claims — linkage is via soft match on (patient_id, cpt_code, date range).

In [0]:
# ============================================================
# AUTHORIZATION GENERATION (~15,000 rows)
# No hard FK to claims - soft match gap causes CO-197 denials
# ============================================================

# CPT codes that require prior auth (advanced imaging)
AUTH_CPTS = {
    "MRI": ["70553", "72148", "73721", "70551", "73718"],
    "CT": ["74177", "72131", "74178", "71260"],
    "NM": ["78452", "78816", "78815"],
    "IR": ["36247", "37228", "49083"],
}
all_auth_cpts = [cpt for cpts in AUTH_CPTS.values() for cpt in cpts]

# Auth status: approved 75%, expired 15%, denied 5%, pending 5%
auth_status_weights = ["approved"] * 75 + ["expired"] * 15 + ["denied"] * 5 + ["pending"] * 5

# Commercial payers that require auth (Medicare/Medicaid less commonly)
auth_payer_ids = ["PAY-001", "PAY-002", "PAY-003", "PAY-005", "PAY-006", "PAY-011"]

authorizations = []

for i in range(1, 15001):
    auth_id = f"AUTH-{str(i).zfill(5)}"
    patient_id = random.choice(patient_ids)
    payer_id = random.choice(auth_payer_ids)
    cpt_code = random.choice(all_auth_cpts)
    auth_number = f"AUTH{random.randint(100000, 999999)}"
    status = random.choice(auth_status_weights)
    
    # Requested date: within our data window
    requested_date = START_DATE + timedelta(days=random.randint(0, date_range_days - 14))
    
    approved_date = None
    expiration_date = None
    authorized_units = None
    
    if status == "approved":
        approved_date = requested_date + timedelta(days=random.randint(1, 5))
        expiration_date = approved_date + timedelta(days=90)
        authorized_units = random.choice([1, 1, 1, 2, 3])
    elif status == "expired":
        approved_date = requested_date + timedelta(days=random.randint(1, 3))
        expiration_date = approved_date + timedelta(days=90)  # Already past
        authorized_units = random.choice([1, 2])
    elif status == "pending":
        authorized_units = random.choice([1, 2])
    # denied: no approved_date, no expiration, no units
    
    authorizations.append((
        auth_id, patient_id, payer_id, cpt_code, auth_number,
        status, requested_date, approved_date, expiration_date, authorized_units
    ))

print(f"Authorizations generated: {len(authorizations):,}")
print(f"  - Approved: {sum(1 for a in authorizations if a[5] == 'approved'):,}")
print(f"  - Expired: {sum(1 for a in authorizations if a[5] == 'expired'):,}")
print(f"  - Denied: {sum(1 for a in authorizations if a[5] == 'denied'):,}")
print(f"  - Pending: {sum(1 for a in authorizations if a[5] == 'pending'):,}")

schema = StructType([
    StructField("auth_id", StringType()),
    StructField("patient_id", StringType()),
    StructField("payer_id", StringType()),
    StructField("cpt_code", StringType()),
    StructField("auth_number", StringType()),
    StructField("auth_status", StringType()),
    StructField("requested_date", DateType()),
    StructField("approved_date", DateType()),
    StructField("expiration_date", DateType()),
    StructField("authorized_units", IntegerType()),
])

df_auths = spark.createDataFrame(authorizations, schema=schema)
df_auths.write.mode("overwrite").saveAsTable("authorization")
print(f"\nauthorization written: {df_auths.count():,} rows")

## 8. Verification

Row counts and DQ trap spot-checks.

In [0]:
# Final row counts for all tables
print("=" * 60)
print("RADIOLOGY REVENUE CYCLE — Final Table Counts")
print("=" * 60)

tables = ["patient", "provider", "service_location", "payer", "encounter", 
          "claim_header", "claim_line", "denial", "payment", "ar_aging_snapshot", "authorization"]

total_rows = 0
for t in tables:
    count = spark.table(t).count()
    total_rows += count
    print(f"  {t:.<30} {count:>10,} rows")

print(f"{'':.<30} {'':->14}")
print(f"  {'TOTAL':.<30} {total_rows:>10,} rows")

In [0]:
# ============================================================
# VERIFY ALL EIGHT TRUST TRAPS ARE SEEDED
# ============================================================
print("=" * 70)
print("TRUST TRAP VERIFICATION (8 traps across 6 dimensions)")
print("=" * 70)

# 1. COMPLETENESS: NULL allowed_amount on paid claims
print("\n1. COMPLETENESS TRAP: NULL allowed_amount on paid claims")
result = spark.sql("""
  SELECT 
    COUNT(*) as total_paid,
    SUM(CASE WHEN total_allowed_amount IS NULL THEN 1 ELSE 0 END) as null_allowed,
    ROUND(SUM(CASE WHEN total_allowed_amount IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as pct_null
  FROM claim_header WHERE claim_status = 'paid'
""")
result.display()

# 2. VALIDITY: Invalid CARC codes
print("\n2. VALIDITY TRAP: Invalid CARC codes in denials")
result = spark.sql("""
  SELECT carc_code, COUNT(*) as cnt,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM denial), 1) as pct_of_total
  FROM denial 
  WHERE carc_code IN ('CO-999', 'PI-300', 'OA-500', 'PR-ZZZ')
  GROUP BY carc_code ORDER BY cnt DESC
""")
result.display()

# 3. CONSISTENCY: Payer name variants
print("\n3. CONSISTENCY TRAP: Payer name variants per payer_id")
result = spark.sql("""
  SELECT ch.payer_id, p.payer_name as canonical_name, 
    COUNT(DISTINCT ch.payer_name) as variant_count,
    COLLECT_SET(ch.payer_name) as variants
  FROM claim_header ch JOIN payer p ON ch.payer_id = p.payer_id
  GROUP BY ch.payer_id, p.payer_name
  HAVING COUNT(DISTINCT ch.payer_name) > 1
  ORDER BY variant_count DESC
  LIMIT 5
""")
result.display()

# 4. UNIQUENESS: Rebill duplicates
print("\n4. UNIQUENESS TRAP: Rebill duplicates (same patient + encounter)")
result = spark.sql("""
  SELECT 
    COUNT(*) as total_claims,
    SUM(CASE WHEN is_rebill = true THEN 1 ELSE 0 END) as flagged_rebills,
    (SELECT COUNT(*) FROM (
      SELECT encounter_id, patient_id, total_charge_amount
      FROM claim_header
      GROUP BY encounter_id, patient_id, total_charge_amount
      HAVING COUNT(*) > 1
    )) as duplicate_groups
  FROM claim_header
""")
result.display()

# 5. TIMELINESS: AR snapshot staleness
print("\n5. TIMELINESS TRAP: AR snapshot freshness by source")
result = spark.sql("""
  SELECT snapshot_source, 
    MIN(snapshot_date) as earliest,
    MAX(snapshot_date) as latest,
    COUNT(DISTINCT snapshot_date) as days_of_data,
    DATEDIFF(DATE '2026-05-15', MAX(snapshot_date)) as days_stale
  FROM ar_aging_snapshot
  GROUP BY snapshot_source
""")
result.display()

# 6. ACCURACY (A): TC/26 component overlap
print("\n6. ACCURACY (A): TC/26 Component Overlap with Global charges")
result = spark.sql("""
  SELECT 
    COUNT(DISTINCT cl_global.claim_id) AS claims_with_overlap,
    COUNT(*) AS overlapping_line_pairs,
    ROUND(SUM(cl_global.charge_amount), 2) AS double_counted_dollars
  FROM claim_line cl_global
  JOIN claim_line cl_component 
    ON cl_global.claim_id = cl_component.claim_id
    AND cl_global.cpt_code = cl_component.cpt_code
    AND cl_global.claim_line_id != cl_component.claim_line_id
  WHERE cl_global.modifier_1 IS NULL
    AND cl_component.modifier_1 IN ('26', 'TC')
""")
result.display()

# 7. ACCURACY (B): allowed = billed for PAY-011 / PAY-012
print("\n7. ACCURACY (B): Clearinghouse error (allowed = charge) for PAY-011/PAY-012")
result = spark.sql("""
  SELECT 
    ch.payer_id,
    p.payer_name,
    COUNT(*) as total_lines,
    SUM(CASE WHEN cl.allowed_amount = cl.charge_amount THEN 1 ELSE 0 END) as allowed_equals_charge,
    ROUND(100.0 * SUM(CASE WHEN cl.allowed_amount = cl.charge_amount THEN 1 ELSE 0 END) / COUNT(*), 1) as pct
  FROM claim_line cl
  JOIN claim_header ch ON cl.claim_id = ch.claim_id
  JOIN payer p ON ch.payer_id = p.payer_id
  WHERE ch.payer_id IN ('PAY-011', 'PAY-012')
    AND cl.allowed_amount IS NOT NULL
  GROUP BY ch.payer_id, p.payer_name
""")
result.display()

# 8. ACCURACY (C): BCBS stale fee schedule (payment > allowed by ~12%)
print("\n8. ACCURACY (C): BCBS Q1 2026 stale fee schedule (payment exceeds allowed)")
result = spark.sql("""
  SELECT 
    COUNT(*) as bcbs_q1_lines,
    ROUND(AVG((cl.paid_amount - cl.allowed_amount) / cl.allowed_amount * 100), 1) as avg_overpayment_pct,
    ROUND(SUM(cl.paid_amount - cl.allowed_amount), 2) as total_understated_revenue
  FROM claim_line cl
  JOIN claim_header ch ON cl.claim_id = ch.claim_id
  WHERE ch.payer_id = 'PAY-001'
    AND cl.service_date BETWEEN '2026-01-01' AND '2026-02-28'
    AND cl.allowed_amount IS NOT NULL
    AND cl.paid_amount IS NOT NULL
    AND cl.paid_amount > cl.allowed_amount * 1.05
""")
result.display()

print("\n" + "=" * 70)
print("ALL EIGHT TRAPS VERIFIED \u2714")
print("=" * 70)